In [1]:
import copy
import torch 
import pandas as pd
import numpy as np

from pathlib import Path
from tqdm import tqdm 

from neuralhydrology.datasetzoo import get_dataset
from neuralhydrology.evaluation import get_tester
from neuralhydrology.datautils.utils import load_scaler
from neuralhydrology.modelzoo.ealstm import EALSTM
from neuralhydrology.utils.config import Config

In [2]:
torch.cuda.get_device_name()

'NVIDIA A40'

In [ ]:
# Load the config.
run_dir_path = Path("/home/wuhlmann/BA/test_runs/runs/full_q_512_3011_185525")
cfg = Config(run_dir_path / "config.yml")


In [ ]:
# Load the final model configuration.
ea_lstm = EALSTM(cfg=cfg)
final_weigthts = torch.load(str(run_dir_path / "model_epoch030.pt"), map_location="cpu")
ea_lstm.load_state_dict(final_weigthts)

<All keys matched successfully>

In [5]:
# Load dataset (for now just test (later combine all sets)).
scaler = load_scaler(run_dir=run_dir_path)
ds = get_dataset(cfg=cfg, is_train=True, period="validation", scaler=scaler)

100%|██████████| 86/86 [00:05<00:00, 15.32it/s]


In [6]:
#attribute one. 
tester = get_tester(cfg=cfg, run_dir=run_dir_path, period="validation", init_model=True)

In [7]:
raw_results = tester.evaluate(save_results=False, metrics=["NSE"])

# Validation: 100%|██████████| 86/86 [01:47<00:00,  1.25s/it]


Experiment with noise on slope. 

In [8]:
basin_ids_list = list(tester.cached_datasets.keys())
attr_ids = [0, 1]
noise_amounts = [-0.1, 0.1]

num_basins = len(basin_ids_list)
num_attr = len(attr_ids)
noise_levels = len(noise_amounts)

# 3D array with dim basin x attribute x noise_level, to hold raw numeric values
raw_array = np.zeros([num_basins, num_attr, noise_levels])

for attr_id in attr_ids:
	
	for noise_id in range(noise_levels): 

		# restore original attributes values in tester
		tester_copy = copy.deepcopy(tester)

		# add noise to the attribute in every catchment
		for id in basin_ids_list:	
			tester_copy.cached_datasets[id]._attributes[id][attr_id] += noise_amounts[noise_id]

		# run evaluation for the currrent noise level
		tester_result_dict = tester_copy.evaluate(save_results=False, metrics=["NSE"])
		nse_values = [tester_result_dict[id]["1D"]["NSE"] for id in basin_ids_list]

		# write NSE for all basins, for the current attribute and noise level	
		raw_array[:, attr_id, noise_id] = nse_values	
	

# Validation: 100%|██████████| 86/86 [01:38<00:00,  1.15s/it]


In [13]:
cfg.static_attributes[:2]

['area_calc', 'elev_mean']

In [18]:
results_dict = {}

for i in range(len(basin_ids_list)): 
    
	results_dict[basin_ids_list[i]] = pd.DataFrame(data=raw_array[i,:,:], index=cfg.static_attributes[:2], columns=noise_amounts)
    

In [23]:
results_dict["30"]

,-0.1,0.1
area_calc,0.502572,0.518110
elev_mean,0.512194,0.513175
